# Evidence-Grounded Factoid SFT for Qwen2.5-0.5B (LoRA rank 16)

Set `USE_SFT_DPO_SPLIT=True` to train on the SFT side of a deterministic question-level 80/20 split of the 1,130 gold-supported questions. The SFT side contains 904 questions expanded into 1,085 supported-alias rows; the other 226 questions are reserved for DPO candidate generation. Set it to `False` to use the original full gold-supported SFT data. Each choice has a separate output directory. Generated-dev evaluation still uses all 160 development prompts.


In [ ]:
from __future__ import annotations
import hashlib
import json
import sys
from pathlib import Path

def project_root(start: Path) -> Path:
    for candidate in [start, *start.parents, start / 'Task-Structured Counterfactual Preference Mining']:
        if (candidate / 'src').is_dir() and (candidate / 'cse_dpo').is_dir(): return candidate.resolve()
    raise RuntimeError('Run from the project or workspace directory.')

PROJECT_ROOT=project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from cse_dpo.normalize_set_answers import normalize_answer_surface
from src.utility.bioasq_format import parse_prediction_items
from src.utility.data import list_record_resources
SOURCE_DIR=PROJECT_ROOT/'data/BioASQ_factoid_sft_prepared/single_answer_full_resources_qwen25_05b'
TRAIN_SOURCE=SOURCE_DIR/'train_prepared.json'
DEV_SOURCE=SOURCE_DIR/'eval_prepared.json'
# Choose 'evidence_all_supported' (one row per question), 'evidence_single_supported_alias' (977 singleton questions), or 'evidence_per_supported_alias' (one row per supported alias).
TRAINING_DATA_VARIANT='evidence_per_supported_alias'
USE_SFT_DPO_SPLIT=True  # True: 904 SFT questions; False: all 1,130 supported questions.
#'factoid-single-answer-extractive-v1'
# The literal-copy run keeps all other settings fixed and changes only the factoid instruction.
PROMPT_ID='factoid-single-answer-extractive-v1'
LORA_R=32
LORA_ALPHA=32  # Preserve alpha/r = 1 from the earlier rank-32/alpha-32 run.
LORA_DROPOUT=0.05
# Shared generated-dev protocol used by both SFT checkpoint selection and DPO evaluation.
STANDARD_EVAL_MAX_SEQ_LENGTH=4096
STANDARD_EVAL_MAX_NEW_TOKENS=64
ALL_SUPPORTED_DIR=PROJECT_ROOT/'data/BioASQ_factoid_sft_prepared/evidence_grounded_single_answer_qwen25_05b'
SINGLE_ALIAS_DIR=PROJECT_ROOT/'data/BioASQ_factoid_sft_prepared/evidence_grounded_single_alias_qwen25_05b'
PER_ALIAS_DIR=PROJECT_ROOT/'data/BioASQ_factoid_sft_prepared/evidence_grounded_per_supported_alias_qwen25_05b'
PREPARED_DIR={'evidence_all_supported': ALL_SUPPORTED_DIR, 'evidence_single_supported_alias': SINGLE_ALIAS_DIR, 'evidence_per_supported_alias': PER_ALIAS_DIR}[TRAINING_DATA_VARIANT]
FULL_FILTERED_TRAIN=PREPARED_DIR/'train_prepared.json'
DEV_SUPPORTED_IDS=PREPARED_DIR/'dev_supported_question_ids.json'
SFT_DPO_SPLIT_DIR=PROJECT_ROOT/'data/BioASQ_factoid_sft_prepared/gold_supported_question_split_sft80_dpo20_seed3407'
SFT_SPLIT_TRAIN=SFT_DPO_SPLIT_DIR/'sft_per_supported_alias_train_prepared.json'
FILTERED_TRAIN=SFT_SPLIT_TRAIN if USE_SFT_DPO_SPLIT else FULL_FILTERED_TRAIN
DPO_QUESTION_INPUT=SFT_DPO_SPLIT_DIR/'dpo_questions_for_candidate_generation.json'
SPLIT_MANIFEST_PATH=SFT_DPO_SPLIT_DIR/'manifest.json'

def supported_aliases(row: dict) -> list[str]:
    evidence=normalize_answer_surface(' '.join(list_record_resources(row)))
    aliases=parse_prediction_items(str(row.get('output','')), 'factoid')
    return [alias for alias in aliases if (normalized:=normalize_answer_surface(alias)) and f' {normalized} ' in f' {evidence} ']

train_rows=json.loads(TRAIN_SOURCE.read_text(encoding='utf-8'))
dev_rows=json.loads(DEV_SOURCE.read_text(encoding='utf-8'))
all_supported_train=[]
single_supported_alias_train=[]
per_supported_alias_train=[]
for row in train_rows:
    aliases=supported_aliases(row)
    if aliases:
        filtered=dict(row); filtered['output']=''.join(f'[BE]{alias}[EE]' for alias in aliases); all_supported_train.append(filtered)
        if len(aliases)==1:
            single_supported_alias_train.append(filtered)
        for alias_index,alias in enumerate(aliases):
            per_alias=dict(row); per_alias['id']=f"{row['id']}__supported_alias_{alias_index}"; per_alias['source_question_id']=row['id']; per_alias['output']=f'[BE]{alias}[EE]'; per_supported_alias_train.append(per_alias)
dev_supported_ids=[row['id'] for row in dev_rows if supported_aliases(row)]
ALL_SUPPORTED_DIR.mkdir(parents=True,exist_ok=True)
SINGLE_ALIAS_DIR.mkdir(parents=True,exist_ok=True)
PER_ALIAS_DIR.mkdir(parents=True,exist_ok=True)
(ALL_SUPPORTED_DIR/'train_prepared.json').write_text(json.dumps(all_supported_train,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
(SINGLE_ALIAS_DIR/'train_prepared.json').write_text(json.dumps(single_supported_alias_train,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
(PER_ALIAS_DIR/'train_prepared.json').write_text(json.dumps(per_supported_alias_train,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
DEV_SUPPORTED_IDS.write_text(json.dumps({'question_ids':dev_supported_ids},indent=2)+'\n',encoding='utf-8')
assert len(all_supported_train)==1130, len(all_supported_train)
assert len(single_supported_alias_train)==977, len(single_supported_alias_train)
assert len(per_supported_alias_train)==1352, len(per_supported_alias_train)
assert all(len(parse_prediction_items(row['output'], 'factoid'))==1 for row in per_supported_alias_train)
assert len(dev_supported_ids)==129, len(dev_supported_ids)
split_manifest=json.loads(SPLIT_MANIFEST_PATH.read_text(encoding='utf-8'))
assert split_manifest['sft_question_count']==904, split_manifest
assert split_manifest['dpo_question_count']==226, split_manifest
assert split_manifest['sft_expanded_alias_row_count']==1085, split_manifest
assert split_manifest['disjoint'] and split_manifest['exhaustive']
assert hashlib.sha256((ALL_SUPPORTED_DIR/'train_prepared.json').read_bytes()).hexdigest()==split_manifest['source_files']['question_source_sha256'], 'Question-level source changed; rebuild the SFT/DPO split.'
assert hashlib.sha256((PER_ALIAS_DIR/'train_prepared.json').read_bytes()).hexdigest()==split_manifest['source_files']['expanded_alias_source_sha256'], 'Alias-expanded source changed; rebuild the SFT/DPO split.'
active_train_rows=json.loads(FILTERED_TRAIN.read_text(encoding='utf-8'))
dpo_question_rows=json.loads(DPO_QUESTION_INPUT.read_text(encoding='utf-8'))
assert len(dpo_question_rows)==226, len(dpo_question_rows)
active_question_ids={row.get('source_question_id',row['id']) for row in active_train_rows}
if USE_SFT_DPO_SPLIT:
    assert TRAINING_DATA_VARIANT=='evidence_per_supported_alias', 'The question-level split currently supplies the per-supported-alias SFT variant.'
    assert len(active_train_rows)==1085, len(active_train_rows)
    assert len(active_question_ids)==904, len(active_question_ids)
    assert not (active_question_ids & {row['id'] for row in dpo_question_rows})
else:
    expected_full={'evidence_all_supported':all_supported_train, 'evidence_single_supported_alias':single_supported_alias_train, 'evidence_per_supported_alias':per_supported_alias_train}[TRAINING_DATA_VARIANT]
    assert active_train_rows==expected_full
print(f'Training variant: {TRAINING_DATA_VARIANT}')
print(f'LoRA rank/alpha/dropout: {LORA_R}/{LORA_ALPHA}/{LORA_DROPOUT}')
print(f'All evidence-supported train: {len(all_supported_train)} / {len(train_rows)}')
print(f'Single-supported-alias train: {len(single_supported_alias_train)} / {len(train_rows)}')
print(f'Per-supported-alias train: {len(per_supported_alias_train)} rows from {len(all_supported_train)} questions')
print(f'Use SFT/DPO split: {USE_SFT_DPO_SPLIT}')
print(f'Active SFT input: {len(active_train_rows)} rows from {len(active_question_ids)} questions')
dpo_status='held out from SFT' if USE_SFT_DPO_SPLIT else 'included in the full SFT data'
print(f'DPO question list: {len(dpo_question_rows)} questions ({dpo_status})')
print(f'Supported dev: {len(dev_supported_ids)} / {len(dev_rows)}')


## Train

Checkpoint selection saves the best all-dev MRR adapter and a separate best evidence-supported MRR adapter. Early stopping treats improvement in either generated MRR, or eval loss, as progress. Rank-16 SFT-split checkpoints are isolated from both the earlier rank-32 run and the full-data rank-16 run.


In [ ]:
import subprocess
BASE_MODEL_DIR_NAME={'evidence_all_supported':'evidence_grounded_single_answer_qwen25_05b', 'evidence_single_supported_alias':'evidence_grounded_single_alias_qwen25_05b', 'evidence_per_supported_alias':'evidence_grounded_per_supported_alias_qwen25_05b'}[TRAINING_DATA_VARIANT]
LORA_TAG=f'lora_r{LORA_R}_alpha{LORA_ALPHA}'
DROPOUT_TAG=f'dropout_{LORA_DROPOUT:.2f}'.replace('.', '')
SPLIT_TAG='sft80_dpo20_seed3407' if USE_SFT_DPO_SPLIT else 'full_gold_supported'
PROMPT_TAG={
    'factoid-single-answer-aligned-v1': 'aligned',
    'factoid-single-answer-extractive-v1': 'strict_extractive',
    'factoid-single-answer-literal-copy-v1': 'literal_copy',
}[PROMPT_ID]
MODEL_DIR=PROJECT_ROOT/'Artifacts/Factoid_SFT/models'/f'{BASE_MODEL_DIR_NAME}_{SPLIT_TAG}_{LORA_TAG}_{DROPOUT_TAG}_{PROMPT_TAG}'
command=[
 'python','src/utility/answer_gen_ft.py','--train-input',str(FILTERED_TRAIN),'--eval-input',str(DEV_SOURCE),'--question-types','factoid',
 '--prepared-output-dir',str(MODEL_DIR/'trainer_prepared'),'--model-name',str(PROJECT_ROOT/'models/Qwen2.5-0.5B-Instruct'),
 '--chat-template','qwen-2.5','--prompt-format','chat','--prompt-file','prompts/factoid_single_answer_aligned.json','--prompt',PROMPT_ID,
 '--max-resources','0','--max-resource-chars','0','--max-factoid-answers','5','--max-seq-length','4096',
 '--per-device-train-batch-size','1','--per-device-eval-batch-size','1','--gradient-accumulation-steps','32','--warmup-steps','5','--num-train-epochs','6.0','--learning-rate','6e-4','--weight-decay','0.01',
 '--logging-steps','1','--save-strategy','steps','--save-steps','25','--lora-r',str(LORA_R),'--lora-alpha',str(LORA_ALPHA),'--lora-dropout',str(LORA_DROPOUT),'--dataset-num-proc','1',
 '--selection-metric','generated_primary_score','--generated-metric-subset-ids',str(DEV_SUPPORTED_IDS),'--selection-max-seq-length',str(STANDARD_EVAL_MAX_SEQ_LENGTH),'--selection-max-new-tokens',str(STANDARD_EVAL_MAX_NEW_TOKENS),'--selection-num-generations','1','--selection-aggregation-strategy','union',
 '--early-stopping-patience','3','--early-stopping-threshold','0.001','--seed','3407','--run-name',f'bioasq-qwen25-05b-factoid-{TRAINING_DATA_VARIANT}-{PROMPT_TAG}-{SPLIT_TAG}-r{LORA_R}-a{LORA_ALPHA}-dropout-{LORA_DROPOUT:.2f}-sft',
 '--output-dir',str(MODEL_DIR/'trainer_output'),'--save-model-dir',str(MODEL_DIR/'adapter'),'--resume-from-checkpoint','auto','--local-files-only']
print(' '.join(command))
subprocess.run(command,cwd=PROJECT_ROOT,check=True)


## Generated Dev Metrics

Each history row contains all-dev MRR on 160 questions and supported-dev MRR on 129 questions. Both come from one decoding pass.


In [ ]:
history_path=MODEL_DIR/'generated_dev_selection/history.json'
best_path=MODEL_DIR/'generated_dev_selection/best_summary.json'
evidence_best_path=MODEL_DIR/'evidence_mrr_selection/best_summary.json'
assert history_path.exists(), f'Missing history: {history_path}'
for entry in json.loads(history_path.read_text(encoding='utf-8')):
    print({'step':entry.get('step'),'epoch':entry.get('epoch'),'all_dev_mrr_160':entry.get('metric_value'),'supported_dev_mrr_129':entry.get('generated_subset_mrr'),'supported_dev_questions':entry.get('generated_subset_question_count')})
if best_path.exists():
    best=json.loads(best_path.read_text(encoding='utf-8'))
    print('Best all-dev checkpoint:',{'step':best.get('step'),'all_dev_mrr_160':best.get('metric_value'),'supported_dev_mrr_129':best.get('generated_subset_mrr')})
if evidence_best_path.exists():
    best=json.loads(evidence_best_path.read_text(encoding='utf-8'))
    print('Best evidence-supported checkpoint:',{'step':best.get('step'),'all_dev_mrr_160':best.get('metric_value'),'supported_dev_mrr_129':best.get('generated_subset_mrr')})
